In [1]:
import sys
sys.path.append('.')
import samna
import numpy as np
import matplotlib

matplotlib.use("TkAgg")          # Or "Qt5Agg", "MacOSX", "WebAgg"
import matplotlib.pyplot as plt
import samna.dynapse1 as dyn1

import dynapse1utils as ut
from netgen import Neuron, NetworkGenerator
from params_all_cores import *
import time
import importlib
from collections import deque
import threading
sys.path.append('../Tools')
from dynamicPlottingTools import *

In [2]:
# to be used when connecting to Dynap-se locally 
devices = samna.device.get_unopened_devices()
print("Available devices: ", devices)
model = samna.device.open_device(devices[0])

Available devices:  [device::DeviceInfo(serial_number=00000033, usb_bus_number=3, usb_device_address=14, logic_version=5, device_type_name=Dynapse1DevKit)]


In [3]:
# Create data buffers
spike_buffer = deque(maxlen=500)
voltage_buffer = deque(maxlen=500)

In [4]:
num_neurons = 10
positions = np.linspace(0, 2*pi, num_neurons, endpoint=False)

# Create plot manager
manager = DynamicPlotManager(update_interval=50)

# Add plots with method chaining
manager.add_plot(
    DynamicRasterPlot,
    data_buffer=spike_buffer,
    num_neurons=num_neurons,
    duration_window=2.0
# ).add_plot(
#     DynamicMembraneTraces,
#     data_buffer=voltage_buffer,
#     neurons_to_plot=[0, 2, 4, 6, 8],
#     time_window=1.0,
#     Vth=-50*mV
).add_plot(
    DynamicPVAPlot,
    data_buffer=spike_buffer,
    positions=positions,
    num_neurons=num_neurons,
    time_window=2.0
)

# Setup and show plots
manager.setup().show(block=False)

WARNING    /home/bmaacaron-iit.local/Documents/Git Repos/JointAttractorNets/dynap-se1/../Tools/dynamicPlottingTools.py:53: UserWarning: frames=None which we can infer the length of, did not pass an explicit *save_count* and passed cache_frame_data=True.  To avoid a possibly unbounded cache, frame data caching has been disabled. To suppress this warning either pass `cache_frame_data=False` or `save_count=MAX_FRAMES`.
  self.animation = FuncAnimation(self.fig, self.update,
 [py.warnings]


In [5]:
def dynapse_run():  

    def gen_clean_param_group():
        """Generate a Dynapse1ParameterGroup of one core which should 
        be able to silence the neurons.

        Returns:
            samna.dynapse1.Dynapse1ParameterGroup: Dynapse1ParameterGroup.
        """
        param_group = dyn1.Dynapse1ParameterGroup()
        # THR
        # ok
        param_group.param_map["IF_THR_N"].coarse_value = 5
        param_group.param_map["IF_THR_N"].fine_value = 80

        # refactory period
        param_group.param_map["IF_RFR_N"].coarse_value = 4
        param_group.param_map["IF_RFR_N"].fine_value = 128

        # leakage
        param_group.param_map["IF_TAU1_N"].coarse_value = 4
        param_group.param_map["IF_TAU1_N"].fine_value = 80

        param_group.param_map["IF_TAU2_N"].coarse_value = 7
        param_group.param_map["IF_TAU2_N"].fine_value = 255

        param_group.param_map["IF_DC_P"].coarse_value = 0
        param_group.param_map["IF_DC_P"].fine_value = 0

        param_group.param_map["NPDPIE_TAU_F_P"].coarse_value = 4
        param_group.param_map["NPDPIE_TAU_F_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_F_P"].coarse_value = 0
        param_group.param_map["NPDPIE_THR_F_P"].fine_value = 0

        param_group.param_map["PS_WEIGHT_EXC_F_N"].coarse_value = 0
        param_group.param_map["PS_WEIGHT_EXC_F_N"].fine_value = 0

        param_group.param_map["NPDPIE_TAU_S_P"].coarse_value = 4
        param_group.param_map["NPDPIE_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_S_P"].coarse_value = 0
        param_group.param_map["NPDPIE_THR_S_P"].fine_value = 0

        param_group.param_map["PS_WEIGHT_EXC_S_N"].coarse_value = 0
        param_group.param_map["PS_WEIGHT_EXC_S_N"].fine_value = 0

        param_group.param_map["IF_NMDA_N"].coarse_value = 0
        param_group.param_map["IF_NMDA_N"].fine_value = 0

        param_group.param_map["NPDPII_TAU_F_P"].coarse_value = 4
        param_group.param_map["NPDPII_TAU_F_P"].fine_value = 80

        param_group.param_map["NPDPII_THR_F_P"].coarse_value = 0
        param_group.param_map["NPDPII_THR_F_P"].fine_value = 0

        param_group.param_map["PS_WEIGHT_INH_F_N"].coarse_value = 0
        param_group.param_map["PS_WEIGHT_INH_F_N"].fine_value = 0

        param_group.param_map["NPDPII_TAU_S_P"].coarse_value = 4 # gaba b synaptic time constant
        param_group.param_map["NPDPII_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPII_THR_S_P"].coarse_value = 0 # gaba b synaptic threshold
        param_group.param_map["NPDPII_THR_S_P"].fine_value = 0

        param_group.param_map["PS_WEIGHT_INH_S_N"].coarse_value = 0 # gaba b synaptic weight
        param_group.param_map["PS_WEIGHT_INH_S_N"].fine_value = 0

        param_group.param_map["IF_AHTAU_N"].coarse_value = 4
        param_group.param_map["IF_AHTAU_N"].fine_value = 80

        param_group.param_map["IF_AHTHR_N"].coarse_value = 0
        param_group.param_map["IF_AHTHR_N"].fine_value = 0

        param_group.param_map["IF_AHW_P"].coarse_value = 0
        param_group.param_map["IF_AHW_P"].fine_value = 0

        param_group.param_map["IF_CASC_N"].coarse_value = 0
        param_group.param_map["IF_CASC_N"].fine_value = 0

        param_group.param_map["PULSE_PWLK_P"].coarse_value = 4
        param_group.param_map["PULSE_PWLK_P"].fine_value = 106

        param_group.param_map["R2R_P"].coarse_value = 3
        param_group.param_map["R2R_P"].fine_value = 85

        param_group.param_map["IF_BUF_P"].coarse_value = 3
        param_group.param_map["IF_BUF_P"].fine_value = 80

        return param_group

    def gen_param_group_c0():
        """Generate a Dynapse1ParameterGroup of one core with some synapse
        weights turned on for examples.

        Returns:
            samna.dynapse1.Dynapse1ParameterGroup: Dynapse1ParameterGroup.
        """
        param_group = dyn1.Dynapse1ParameterGroup()
        # THR
        # ok
        param_group.param_map["IF_THR_N"].coarse_value = 5
        param_group.param_map["IF_THR_N"].fine_value = 80

        # refactory period
        param_group.param_map["IF_RFR_N"].coarse_value = 4
        param_group.param_map["IF_RFR_N"].fine_value = 128

        # leakage
        param_group.param_map["IF_TAU1_N"].coarse_value = 2 # was 4
        param_group.param_map["IF_TAU1_N"].fine_value = 60 # was 120

            # Main neuron time constant (unless switched to TAU2)

            # neuron time constant = how quickly neuron's membrane potential changes in response to inputs
            # short time constant --> neuron more sensitive to rapid changes, less responsive to sustained inputs
    
        param_group.param_map["IF_TAU2_N"].coarse_value = 7
        param_group.param_map["IF_TAU2_N"].fine_value = 255

        param_group.param_map["IF_DC_P"].coarse_value = 0
        param_group.param_map["IF_DC_P"].fine_value = 0

        param_group.param_map["NPDPIE_TAU_F_P"].coarse_value = 5
        param_group.param_map["NPDPIE_TAU_F_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_F_P"].coarse_value = 4
        param_group.param_map["NPDPIE_THR_F_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_F_N"].coarse_value = 6 #trying new values to test with Mirco's chip # was 6
        param_group.param_map["PS_WEIGHT_EXC_F_N"].fine_value = 40 # was 40
        
            # Fast excitatory (AMPA) synapse weights
            # sets weight of the fast excitatory synapses
            # determines strength of synaptic input to the neuron
            # increasing it = increase strength of excitatory input, can increase network's overall activity and excitability

        param_group.param_map["NPDPIE_TAU_S_P"].coarse_value = 4
        param_group.param_map["NPDPIE_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_S_P"].coarse_value = 4
        param_group.param_map["NPDPIE_THR_S_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_S_N"].coarse_value = 6 # 6.80 per EE
        param_group.param_map["PS_WEIGHT_EXC_S_N"].fine_value = 30

        param_group.param_map["IF_NMDA_N"].coarse_value = 0
        param_group.param_map["IF_NMDA_N"].fine_value = 0

        param_group.param_map["NPDPII_TAU_F_P"].coarse_value = 3
        param_group.param_map["NPDPII_TAU_F_P"].fine_value = 80
            # Fast inhibitory (GABA_A) synapses time constant
            # Affects how quickly inhibitory currents decay.
            # increasing: inhibitory effect lasts longer - can lead to more prolonged inhibition, potentially suppressing network activity more effectively
            # decreasing: shorten time constant, inhibitory effects decay faster - can reduce duration of inhibition, potentially allowing network to recover more quickly from inhibitory events

        param_group.param_map["NPDPII_THR_F_P"].coarse_value = 5 # was 5
        param_group.param_map["NPDPII_THR_F_P"].fine_value = 80 # was 80
        
            # Fast inhibitory (GABA_A) synapses threshold ((i.e. max I_syn value)
            # Sets threshold for fast inhibitory synapses, affecting max inhibitory synaptic current.

        param_group.param_map["PS_WEIGHT_INH_F_N"].coarse_value = 6 # was 6
        param_group.param_map["PS_WEIGHT_INH_F_N"].fine_value = 50 # was 50

        param_group.param_map["NPDPII_TAU_S_P"].coarse_value = 3 # gaba b synaptic time constant - slow inhibitory
        param_group.param_map["NPDPII_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPII_THR_S_P"].coarse_value = 4 # gaba b synaptic threshold  # mirco modification # was 4
        param_group.param_map["NPDPII_THR_S_P"].fine_value = 80 # mirco modification  # was 80

        param_group.param_map["PS_WEIGHT_INH_S_N"].coarse_value = 7 # gaba B synaptic weight # making this stronger to use with Mirco's chip # before it was 7 coarse, 20 fine
        param_group.param_map["PS_WEIGHT_INH_S_N"].fine_value = 20

        param_group.param_map["IF_AHTAU_N"].coarse_value = 4
        param_group.param_map["IF_AHTAU_N"].fine_value = 80

        param_group.param_map["IF_AHTHR_N"].coarse_value = 0
        param_group.param_map["IF_AHTHR_N"].fine_value = 0

        param_group.param_map["IF_AHW_P"].coarse_value = 0
        param_group.param_map["IF_AHW_P"].fine_value = 0

        param_group.param_map["IF_CASC_N"].coarse_value = 0
        param_group.param_map["IF_CASC_N"].fine_value = 0

        param_group.param_map["PULSE_PWLK_P"].coarse_value = 4
        param_group.param_map["PULSE_PWLK_P"].fine_value = 106

        param_group.param_map["R2R_P"].coarse_value = 3
        param_group.param_map["R2R_P"].fine_value = 85

        param_group.param_map["IF_BUF_P"].coarse_value = 3
        param_group.param_map["IF_BUF_P"].fine_value = 80

        return param_group

    def gen_param_group_c1():
        """Generate a Dynapse1ParameterGroup of one core with some synapse
        weights turned on for examples.

        Returns:
            samna.dynapse1.Dynapse1ParameterGroup: Dynapse1ParameterGroup.
        """
        """    param_group = dyn1.Dynapse1ParameterGroup()
        # THR
        # ok
        param_group.param_map["IF_THR_N"].coarse_value = 3
        param_group.param_map["IF_THR_N"].fine_value = 80

        # refactory period
        param_group.param_map["IF_RFR_N"].coarse_value = 4
        param_group.param_map["IF_RFR_N"].fine_value = 128

        # leakage
        param_group.param_map["IF_TAU1_N"].coarse_value = 5 # was 4 
        param_group.param_map["IF_TAU1_N"].fine_value = 120 # was 120

            # Main neuron time constant (unless switched to TAU2)

            # neuron time constant = how quickly neuron's membrane potential changes in response to inputs
            # short time constant --> neuron more sensitive to rapid changes, less responsive to sustained inputs
            # Short time constant obtained with higher values of IF_TAU1_N or IF_TAU2_N
    
        param_group.param_map["IF_TAU2_N"].coarse_value = 7
        param_group.param_map["IF_TAU2_N"].fine_value = 255

        param_group.param_map["IF_DC_P"].coarse_value = 0
        param_group.param_map["IF_DC_P"].fine_value = 0
        
        #param_group.param_map["NPDPIE_TAU_F_P"].coarse_value = 1
        #param_group.param_map["NPDPIE_TAU_F_P"].fine_value = 30

        #param_group.param_map["NPDPIE_THR_F_P"].coarse_value = 3
        #param_group.param_map["NPDPIE_THR_F_P"].fine_value = 50
        

        param_group.param_map["NPDPIE_TAU_F_P"].coarse_value = 1 # was 5
        param_group.param_map["NPDPIE_TAU_F_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_F_P"].coarse_value = 6 # was 4
        param_group.param_map["NPDPIE_THR_F_P"].fine_value = 100
        
        param_group.param_map["PS_WEIGHT_EXC_F_N"].coarse_value = 6 #trying new values to test with Mirco's chip # was 6 
        param_group.param_map["PS_WEIGHT_EXC_F_N"].fine_value = 100 # was 40
        
            # Fast excitatory (AMPA) synapse weights
            # sets weight of the fast excitatory synapses
            # determines strength of synaptic input to the neuron
            # increasing it = increase strength of excitatory input, can increase network's overall activity and excitability

        param_group.param_map["NPDPIE_TAU_S_P"].coarse_value = 3 # higher so leaks more
        param_group.param_map["NPDPIE_TAU_S_P"].fine_value = 100

        param_group.param_map["NPDPIE_THR_S_P"].coarse_value = 5
        param_group.param_map["NPDPIE_THR_S_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_S_N"].coarse_value = 6 # 6.80 per EE
        param_group.param_map["PS_WEIGHT_EXC_S_N"].fine_value = 40

        param_group.param_map["IF_NMDA_N"].coarse_value = 0
        param_group.param_map["IF_NMDA_N"].fine_value = 0

        param_group.param_map["NPDPII_TAU_F_P"].coarse_value = 3 
        param_group.param_map["NPDPII_TAU_F_P"].fine_value = 80
            # Fast inhibitory (GABA_A) synapses time constant
            # Affects how quickly inhibitory currents decay.
            # increasing: inhibitory effect lasts longer - can lead to more prolonged inhibition, potentially suppressing network activity more effectively
            # decreasing: shorten time constant, inhibitory effects decay faster - can reduce duration of inhibition, potentially allowing network to recover more quickly from inhibitory events

        param_group.param_map["NPDPII_THR_F_P"].coarse_value = 5 # was 5 
        param_group.param_map["NPDPII_THR_F_P"].fine_value = 80 # was 80
        
            # Fast inhibitory (GABA_A) synapses threshold ((i.e. max I_syn value)
            # Sets threshold for fast inhibitory synapses, affecting max inhibitory synaptic current.

        param_group.param_map["PS_WEIGHT_INH_F_N"].coarse_value = 6 # was 6
        param_group.param_map["PS_WEIGHT_INH_F_N"].fine_value = 50 # was 50

        param_group.param_map["NPDPII_TAU_S_P"].coarse_value = 4 #3 # gaba b synaptic time constant - slow inhibitory
        param_group.param_map["NPDPII_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPII_THR_S_P"].coarse_value = 5 # gaba b synaptic threshold  # mirco modification # was 4
        param_group.param_map["NPDPII_THR_S_P"].fine_value = 100 # mirco modification  # was 80

        param_group.param_map["PS_WEIGHT_INH_S_N"].coarse_value = 6 # gaba B synaptic weight # making this stronger to use with Mirco's chip # before it was 7 coarse, 20 fine
        param_group.param_map["PS_WEIGHT_INH_S_N"].fine_value = 80

        param_group.param_map["IF_AHTAU_N"].coarse_value = 4
        param_group.param_map["IF_AHTAU_N"].fine_value = 80

        param_group.param_map["IF_AHTHR_N"].coarse_value = 0
        param_group.param_map["IF_AHTHR_N"].fine_value = 0

        param_group.param_map["IF_AHW_P"].coarse_value = 0
        param_group.param_map["IF_AHW_P"].fine_value = 0

        param_group.param_map["IF_CASC_N"].coarse_value = 0
        param_group.param_map["IF_CASC_N"].fine_value = 0

        param_group.param_map["PULSE_PWLK_P"].coarse_value = 4
        param_group.param_map["PULSE_PWLK_P"].fine_value = 106

        param_group.param_map["R2R_P"].coarse_value = 3
        param_group.param_map["R2R_P"].fine_value = 85

        param_group.param_map["IF_BUF_P"].coarse_value = 3
        param_group.param_map["IF_BUF_P"].fine_value = 80"""

        param_group = dyn1.Dynapse1ParameterGroup()
        # THR
        # ok
        param_group.param_map["IF_THR_N"].coarse_value = 5
        param_group.param_map["IF_THR_N"].fine_value = 80

        # refactory period
        param_group.param_map["IF_RFR_N"].coarse_value = 4
        param_group.param_map["IF_RFR_N"].fine_value = 128

        # leakage
        param_group.param_map["IF_TAU1_N"].coarse_value = 4
        param_group.param_map["IF_TAU1_N"].fine_value = 80

        param_group.param_map["IF_TAU2_N"].coarse_value = 7
        param_group.param_map["IF_TAU2_N"].fine_value = 255

        param_group.param_map["IF_DC_P"].coarse_value = 0
        param_group.param_map["IF_DC_P"].fine_value = 0

        param_group.param_map["NPDPIE_TAU_F_P"].coarse_value = 4
        param_group.param_map["NPDPIE_TAU_F_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_F_P"].coarse_value = 4
        param_group.param_map["NPDPIE_THR_F_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_F_N"].coarse_value = 6
        param_group.param_map["PS_WEIGHT_EXC_F_N"].fine_value = 120

        param_group.param_map["NPDPIE_TAU_S_P"].coarse_value = 4
        param_group.param_map["NPDPIE_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_S_P"].coarse_value = 5 # was 4
        param_group.param_map["NPDPIE_THR_S_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_S_N"].coarse_value = 5 # was 4
        param_group.param_map["PS_WEIGHT_EXC_S_N"].fine_value = 200 # was 80

        param_group.param_map["IF_NMDA_N"].coarse_value = 0
        param_group.param_map["IF_NMDA_N"].fine_value = 0

        param_group.param_map["NPDPII_TAU_F_P"].coarse_value = 4
        param_group.param_map["NPDPII_TAU_F_P"].fine_value = 80

        param_group.param_map["NPDPII_THR_F_P"].coarse_value = 4
        param_group.param_map["NPDPII_THR_F_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_INH_F_N"].coarse_value = 0
        param_group.param_map["PS_WEIGHT_INH_F_N"].fine_value = 0

        param_group.param_map["NPDPII_TAU_S_P"].coarse_value = 4
        param_group.param_map["NPDPII_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPII_THR_S_P"].coarse_value = 4
        param_group.param_map["NPDPII_THR_S_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_INH_S_N"].coarse_value = 5 # was 5
        param_group.param_map["PS_WEIGHT_INH_S_N"].fine_value = 120 # was 100

        param_group.param_map["IF_AHTAU_N"].coarse_value = 4
        param_group.param_map["IF_AHTAU_N"].fine_value = 80

        param_group.param_map["IF_AHTHR_N"].coarse_value = 0
        param_group.param_map["IF_AHTHR_N"].fine_value = 0

        param_group.param_map["IF_AHW_P"].coarse_value = 0
        param_group.param_map["IF_AHW_P"].fine_value = 0

        param_group.param_map["IF_CASC_N"].coarse_value = 0
        param_group.param_map["IF_CASC_N"].fine_value = 0

        param_group.param_map["PULSE_PWLK_P"].coarse_value = 4
        param_group.param_map["PULSE_PWLK_P"].fine_value = 106

        param_group.param_map["R2R_P"].coarse_value = 3
        param_group.param_map["R2R_P"].fine_value = 85

        param_group.param_map["IF_BUF_P"].coarse_value = 3
        param_group.param_map["IF_BUF_P"].fine_value = 80

        return param_group


        return param_group

    def gen_param_group_c2():

        param_group = dyn1.Dynapse1ParameterGroup()
        # THR
        # ok
        param_group.param_map["IF_THR_N"].coarse_value = 5
        param_group.param_map["IF_THR_N"].fine_value = 80

        # refactory period
        param_group.param_map["IF_RFR_N"].coarse_value = 4
        param_group.param_map["IF_RFR_N"].fine_value = 128

        # leakage
        param_group.param_map["IF_TAU1_N"].coarse_value = 2 # was 4 
        param_group.param_map["IF_TAU1_N"].fine_value = 60 # was 120

            # Main neuron time constant (unless switched to TAU2)

            # neuron time constant = how quickly neuron's membrane potential changes in response to inputs
            # short time constant --> neuron more sensitive to rapid changes, less responsive to sustained inputs
    
        param_group.param_map["IF_TAU2_N"].coarse_value = 7
        param_group.param_map["IF_TAU2_N"].fine_value = 255

        param_group.param_map["IF_DC_P"].coarse_value = 0
        param_group.param_map["IF_DC_P"].fine_value = 0

        param_group.param_map["NPDPIE_TAU_F_P"].coarse_value = 5
        param_group.param_map["NPDPIE_TAU_F_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_F_P"].coarse_value = 4
        param_group.param_map["NPDPIE_THR_F_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_F_N"].coarse_value = 6 #6.80 per INP_E
        param_group.param_map["PS_WEIGHT_EXC_F_N"].fine_value = 50 # was 60         # PS_WEIGHT_EXC_F_N: excitatory (AMPA) synapse weights
        
            # PS_WEIGHT_EXC_F_N: AMPA synapse weight. increasing = increase strength of exc input
            
        param_group.param_map["NPDPIE_TAU_S_P"].coarse_value = 4
        param_group.param_map["NPDPIE_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_S_P"].coarse_value = 4
        param_group.param_map["NPDPIE_THR_S_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_S_N"].coarse_value = 6 # 6.80 per EE
        param_group.param_map["PS_WEIGHT_EXC_S_N"].fine_value = 200 # was 60

        param_group.param_map["IF_NMDA_N"].coarse_value = 0
        param_group.param_map["IF_NMDA_N"].fine_value = 0

        param_group.param_map["NPDPII_TAU_F_P"].coarse_value = 3 
        param_group.param_map["NPDPII_TAU_F_P"].fine_value = 80
            # Fast inhibitory (GABA_A) synapses time constant
            # Affects how quickly inhibitory currents decay.
            # increasing: inhibitory effect lasts longer - can lead to more prolonged inhibition, potentially suppressing network activity more effectively
            # decreasing: shorten time constant, inhibitory effects decay faster - can reduce duration of inhibition, potentially allowing network to recover more quickly from inhibitory events

        param_group.param_map["NPDPII_THR_F_P"].coarse_value = 5 # was 5 
        param_group.param_map["NPDPII_THR_F_P"].fine_value = 80 # was 80
        
            # Fast inhibitory (GABA_A) synapses threshold ((i.e. max I_syn value)
            # Sets threshold for fast inhibitory synapses, affecting max inhibitory synaptic current.

        param_group.param_map["PS_WEIGHT_INH_F_N"].coarse_value = 6 # was 6
        param_group.param_map["PS_WEIGHT_INH_F_N"].fine_value = 50 # was 50

        param_group.param_map["NPDPII_TAU_S_P"].coarse_value = 3 # gaba b synaptic time constant
        param_group.param_map["NPDPII_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPII_THR_S_P"].coarse_value = 4 # gaba b synaptic threshold 
        param_group.param_map["NPDPII_THR_S_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_INH_S_N"].coarse_value = 7 # gaba B synaptic weight  # making this stronger to use with Mirco's chip # before it was 7 coarse, 20 fine
        param_group.param_map["PS_WEIGHT_INH_S_N"].fine_value = 20

        param_group.param_map["IF_AHTAU_N"].coarse_value = 4
        param_group.param_map["IF_AHTAU_N"].fine_value = 80

        param_group.param_map["IF_AHTHR_N"].coarse_value = 0
        param_group.param_map["IF_AHTHR_N"].fine_value = 0

        param_group.param_map["IF_AHW_P"].coarse_value = 0
        param_group.param_map["IF_AHW_P"].fine_value = 0

        param_group.param_map["IF_CASC_N"].coarse_value = 0
        param_group.param_map["IF_CASC_N"].fine_value = 0

        param_group.param_map["PULSE_PWLK_P"].coarse_value = 4
        param_group.param_map["PULSE_PWLK_P"].fine_value = 106

        param_group.param_map["R2R_P"].coarse_value = 3
        param_group.param_map["R2R_P"].fine_value = 85

        param_group.param_map["IF_BUF_P"].coarse_value = 3
        param_group.param_map["IF_BUF_P"].fine_value = 80

        return param_group

    def gen_param_group_c3():
        """Generate a Dynapse1ParameterGroup of one core with some synapse
        weights turned on for examples.

        Returns:
            samna.dynapse1.Dynapse1ParameterGroup: Dynapse1ParameterGroup.
        """
        param_group = dyn1.Dynapse1ParameterGroup()
        # THR
        # ok
        param_group.param_map["IF_THR_N"].coarse_value = 5
        param_group.param_map["IF_THR_N"].fine_value = 80

        # refactory period
        param_group.param_map["IF_RFR_N"].coarse_value = 4
        param_group.param_map["IF_RFR_N"].fine_value = 128

        # leakage
        param_group.param_map["IF_TAU1_N"].coarse_value = 2 # was 4 
        param_group.param_map["IF_TAU1_N"].fine_value = 60 # was 120

            # Main neuron time constant (unless switched to TAU2)

            # neuron time constant = affects how quickly the neuron's membrane potential changes in response to inputs
            # shorter time constant --> neuron more sensitive to rapid changes, less responsive to sustained inputs
    
        param_group.param_map["IF_TAU2_N"].coarse_value = 7
        param_group.param_map["IF_TAU2_N"].fine_value = 255

        param_group.param_map["IF_DC_P"].coarse_value = 0# was 4
        param_group.param_map["IF_DC_P"].fine_value = 0 # was 150
    
        param_group.param_map["NPDPIE_TAU_F_P"].coarse_value = 5
        param_group.param_map["NPDPIE_TAU_F_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_F_P"].coarse_value = 4
        param_group.param_map["NPDPIE_THR_F_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_F_N"].coarse_value = 6 #6.80 per INP_E
        param_group.param_map["PS_WEIGHT_EXC_F_N"].fine_value = 20 # was 40 # was 70
        
        
        
            # Fast excitatory (AMPA) synapse weights
            # sets the weight of the fast excitatory synapses
            # determines strength of synaptic input to the neuron
            # increasing it = increase strength of excitatory input, can increase network's overall activity and excitability

        param_group.param_map["NPDPIE_TAU_S_P"].coarse_value = 4
        param_group.param_map["NPDPIE_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPIE_THR_S_P"].coarse_value = 4
        param_group.param_map["NPDPIE_THR_S_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_EXC_S_N"].coarse_value = 6 # 6.80 per EE
        param_group.param_map["PS_WEIGHT_EXC_S_N"].fine_value = 40

        param_group.param_map["IF_NMDA_N"].coarse_value = 6 # enable NMDA gating!
        param_group.param_map["IF_NMDA_N"].fine_value = 200

        param_group.param_map["NPDPII_TAU_F_P"].coarse_value = 2
        param_group.param_map["NPDPII_TAU_F_P"].fine_value = 60
            # Fast inhibitory (GABA_A) synapses time constant
            # Affects how quickly inhibitory currents decay.
            # increasing: inhibitory effect lasts longer - can lead to more prolonged inhibition, potentially suppressing network activity more effectively
            # decreasing: shorten time constant, inhibitory effects decay faster - can reduce duration of inhibition, potentially allowing network to recover more quickly from inhibitory events

        param_group.param_map["NPDPII_THR_F_P"].coarse_value = 5 # was 5 
        param_group.param_map["NPDPII_THR_F_P"].fine_value = 80 # was 80
        
            # Fast inhibitory (GABA_A) synapses threshold ((i.e. max I_syn value)
            # Sets threshold for fast inhibitory synapses, affecting max inhibitory synaptic current.

        param_group.param_map["PS_WEIGHT_INH_F_N"].coarse_value = 6 # was 6
        param_group.param_map["PS_WEIGHT_INH_F_N"].fine_value = 50 # was 50

        param_group.param_map["NPDPII_TAU_S_P"].coarse_value = 3 # gaba b synaptic time constant
        param_group.param_map["NPDPII_TAU_S_P"].fine_value = 80

        param_group.param_map["NPDPII_THR_S_P"].coarse_value = 4 # gaba b synaptic threshold 
        param_group.param_map["NPDPII_THR_S_P"].fine_value = 80

        param_group.param_map["PS_WEIGHT_INH_S_N"].coarse_value = 6 # gaba B synaptic weight  # making this stronger to use with Mirco's chip # before it was 2 coarse, 8 fine
        param_group.param_map["PS_WEIGHT_INH_S_N"].fine_value = 120 # was 8

        param_group.param_map["IF_AHTAU_N"].coarse_value = 4
        param_group.param_map["IF_AHTAU_N"].fine_value = 80

        param_group.param_map["IF_AHTHR_N"].coarse_value = 0
        param_group.param_map["IF_AHTHR_N"].fine_value = 0

        param_group.param_map["IF_AHW_P"].coarse_value = 0
        param_group.param_map["IF_AHW_P"].fine_value = 0

        param_group.param_map["IF_CASC_N"].coarse_value = 0
        param_group.param_map["IF_CASC_N"].fine_value = 0

        param_group.param_map["PULSE_PWLK_P"].coarse_value = 4
        param_group.param_map["PULSE_PWLK_P"].fine_value = 106

        param_group.param_map["R2R_P"].coarse_value = 3
        param_group.param_map["R2R_P"].fine_value = 85

        param_group.param_map["IF_BUF_P"].coarse_value = 3
        param_group.param_map["IF_BUF_P"].fine_value = 80

        return param_group


    def gen_dc_params():
        """Generate a Dynapse1ParameterGroup based on silent neurons, and turned
        DC current on.

        Returns:
            samna.dynapse1.Dynapse1ParameterGroup: Dynapse1ParameterGroup.
        """
        param_group = gen_clean_param_group()

        param_group.param_map["IF_DC_P"].coarse_value = 0 # was 2 
        param_group.param_map["IF_DC_P"].fine_value = 0 # was 150

        return param_group

    def gen_stdp_params():
        """Generate a Dynapse1ParameterGroup for STDP example.

        Returns:
            samna.dynapse1.Dynapse1ParameterGroup: Dynapse1ParameterGroup.
        """
        param_group = gen_param_group_c2()

        param_group.param_map["IF_TAU1_N"].coarse_value = 4
        param_group.param_map["IF_TAU1_N"].fine_value = 80

        param_group.param_map["IF_THR_N"].coarse_value = 4
        param_group.param_map["IF_THR_N"].fine_value = 80

        # NMDA, pre to post neurons
        param_group.param_map["PS_WEIGHT_EXC_S_N"].coarse_value = 7
        param_group.param_map["PS_WEIGHT_EXC_S_N"].fine_value = 80

        # AMPA, spikegen to neurons
        param_group.param_map["PS_WEIGHT_EXC_F_N"].coarse_value = 6
        param_group.param_map["PS_WEIGHT_EXC_F_N"].fine_value = 80

        return param_group

    def set_params(model, dc=False, param_group=None):
        """Set 16 DYNAP-SE1  cores with the same Dynapse1ParameterGroup for examples.

        Args:
            model (samna.dynapse1.Dynapse1Model): Dynapse1Model
            dc (bool, optional): Turn DC on if True. Defaults to False.
            param_group (samna.dynapse1.Dynapse1ParameterGroup, optional): 
                Dynapse1ParameterGroup can be specified. Defaults to None.
        """
        """if param_group is None:
            if dc:
                param_group = gen_dc_params()
            else:
                param_group = gen_param_group()"""
            
        #for chip in [0]:
            #for core in range(4):
        model.update_parameter_group(gen_param_group_c0(), 0, 0)
        model.update_parameter_group(gen_param_group_c1(), 0, 1)
        model.update_parameter_group(gen_param_group_c2(), 0, 2)
        model.update_parameter_group(gen_param_group_c3(), 0, 3)
            
    def set_stdp_params(model):
        """Set 16 DYNAP-SE1 cores with the same Dynapse1ParameterGroup for STDP example.

        Args:
            model (samna.dynapse1.Dynapse1Model): Dynapse1Model
        """
        param_group = gen_stdp_params()

        for chip in range(4):
            for core in range(4):
                model.update_parameter_group(param_group, chip, core)


    while True:
        api = model.get_dynapse1_api()
        config1 = model.get_configuration()
        param_group_c0 = config1.chips[0].cores[0].parameter_group 
        param_group_c1 = config1.chips[0].cores[1].parameter_group 
        param_group_c2 = config1.chips[0].cores[2].parameter_group 
        param_group_c3 = config1.chips[0].cores[3].parameter_group 

        # LIF encoding

        # ----------------  stimulus: a Gaussian bump ----------------
        n_pts     = 1000                 # number of samples
        t_end     = 1.0                  # seconds  (→ dt = 1 ms)
        t         = np.linspace(0, t_end, n_pts, endpoint=False)
        x         = np.linspace(-4, 4, n_pts)
        sigma = 0.6  # Try smaller values: 1.0 (default), 0.5, 0.25, etc.
        gauss = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * (x / sigma)**2)
        I_peak    = 30000000e-12              # 1000 pA

        I         = gauss/gauss.max() * I_peak   # injected current (A)

        # ----------------  LIF neuron parameters ----------------------
        tau_m     = 20e-3                # 20 ms membrane time constant
        R_m       = 100e6                # 100 MΩ  (=> C = tau/R)
        C_m       = tau_m / R_m
        v_rest    = -65e-3               # -65 mV
        v_reset   = -65e-3
        v_thresh  = -50e-3               # spike threshold
        t_ref     = 2e-3                 # 2 ms refractory period
        dt        = t_end / n_pts        # simulation time-step (s)

        # ----------------  simulation loop ----------------------------
        v        = v_rest
        next_ok  = 0.0                   # time when refractory ends
        v_trace  = np.empty(n_pts)
        spikes   = []

        for k in range(n_pts):
            if t[k] >= next_ok:          # not in refractory
                dv = (-(v - v_rest) + R_m * I[k]) / (R_m * C_m) * dt
                v += dv
                if v >= v_thresh:        # spike!
                    spikes.append(t[k])
                    v = v_reset
                    next_ok = t[k] + t_ref
            v_trace[k] = v

        spikes = np.array(spikes)
        spike_times_all = spikes#*1e3
        spike_ids = np.full(len(spikes), 1)
        spikegen_ids = [(0, 1, n) for n in range(10)]


        config1 = model.get_configuration()
        param_group_c0 = config1.chips[0].cores[0].parameter_group 

        p_E_E   = 0.5
        p_mexican = 1
        p_I_I   = 0 #1
        p_E_I   = 0 #0.1
        p_I_E   = 1 #0.4

        # initiate network 
        net_gen = NetworkGenerator()
        net_gen.clear_network()

        # create spikegens, one per ring attractor neural pop 
        spikegen_ids = [(0, 0, n) for n in range(10)]

        spikegens = []
        for spikegen_id in spikegen_ids:
            spikegens.append(Neuron(spikegen_id[0], spikegen_id[1],spikegen_id[2], True))

        # create neuron populations in the ring
        chip = 0
        core = 3
        npop = 3
        NBINS = 10

        # Create neuron populations for each band in core 0
        ring_pops = [
            [Neuron(chip, core, j) for j in range(10 + (i * npop), 10 + ((i + 1) * npop))]
            for i in range(NBINS)
        ]


        # create inhibitory population that connects to all other pops
        core_inh = 1
        start_inh_neuron = 4
        npop_inh = 4
        pop_inhibitory = [Neuron(chip, core_inh, j) for j in range(start_inh_neuron, start_inh_neuron + npop_inh, 1)]

                
        for i in ring_pops[3]:
            net_gen.add_connection(spikegens[1], i, dyn1.Dynapse1SynType.AMPA)
            

        # self excitation in each neural population in the ring: (todo determine if this is needed) 
        for pop in ring_pops:
            for pre in pop:
                for post in pop:
                    if pre is not post and np.random.rand() < p_E_E:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        
        # MEXICAN HAT CONNECTIONS
        OFFSET_1 = (-1, 1)         

        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                for d in OFFSET_1:
                    j = (i + d) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)

        OFFSET_2 = (-2, 2)          # ±3 bins wide “hat”

            # todo excitatory connections to second neighbors


        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                for d in OFFSET_2:
                    j = (i + d) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)

        OFFSET_3 = (-3, 3)          # ±3 bins wide “hat”
            # todo excitatory connections to third neighbors


        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                for d in OFFSET_3:
                    j = (i + d) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

            # todo inhibitory connections to all of the other pops
        OFFSET_inh = (-6, -5, -4, 4, 5, 6)          # all of the pops that are not being excited


        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                for d in OFFSET_inh:
                    j = (i + d) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)

        # INH → EXC  (global inhibition pop to all pops in the ring)
        for inh in pop_inhibitory:
            for pop in ring_pops:
                for exc in pop:
                    net_gen.add_connection(inh, exc, dyn1.Dynapse1SynType.GABA_B)

        # EXC → INH  (drive the global inhibition pop from all pops in the ring)
        for pop in ring_pops:
            for exc in pop:
                for inh in pop_inhibitory:
                    net_gen.add_connection(exc, inh, dyn1.Dynapse1SynType.NMDA)


        # make a dynapse1config using the network
        new_config = net_gen.make_dynapse1_configuration()

        # apply the configuration
        model.apply_configuration(new_config)

        # Set hardware parameters
        set_params(model)

        #for rate in [50]:
        D_stim = 3 # stimulus duration (s)
        D_post_stim = 3 # post stimulus duration (s)
        D = D_stim + D_post_stim

        fpga_spike_gen = model.get_fpga_spike_gen() # set FPGA 

        # get events of selected neurons
        monitored_neurons = [
            (n.chip_id, n.core_id, n.neuron_id)   
            for pop in ring_pops
            for n   in pop
        ]

        monitored_neurons.extend([
            (neuron.chip_id, neuron.core_id, neuron.neuron_id)
            for neuron in pop_inhibitory  
        ])


        graph, filter_node, sink_node = ut.create_neuron_select_graph(model, monitored_neurons)
        graph.start()

        # clear the buffer
        sink_node.get_events()

        # select the neurons to monitor

        filter_node.set_neurons(monitored_neurons)

        api.reset_timestamp()

        time.sleep(0)

        spike_times_s = spike_times_all # spike_times_all is in seconds!!!

        ut.set_fpga_spike_gen(
            fpga_spike_gen,
            spike_times_all,
            spike_ids,
            target_chips=[0] * len(spike_ids),
            isi_base=900,
            repeat_mode=False)

        fpga_spike_gen.start()

        timesleep = 3
        time.sleep(timesleep)

        fpga_spike_gen.stop()

        graph.stop()
        events = sink_node.get_events()

    # NN = 256 # neurons in 1 core
    # # spike_id = []
    # # spike_t  = []
    # # for evt in events:
    # #     #spike_id.append(evt.core_id*NN + evt.neuron_id)
    # #     spike_id.append(evt.neuron_id)
    # #     spike_t.append(evt.timestamp*1e-6)
    # # spike_id = np.array(spike_id)
    # # spike_t  = np.array(spike_t)
    
    # return spike_id, spike_t
    
    # for evt in events:
    #     spike_buffer.append((evt.neuron_id, evt.timestamp*1e-6))
        
        # Collect spikes in a batch
        if len(events) > 0:
            spike_ids = []
            spike_times = []
            for evt in events:
                spike_ids.append(evt.neuron_id)
                spike_times.append(evt.timestamp*1e-6)
            
            # Add the batch as a single entry to buffer
            spike_buffer.append((np.array(spike_ids), np.array(spike_times)))


In [6]:
# Start simulation in a separate thread
dynapse_thread = threading.Thread(target=dynapse_run)
dynapse_thread.daemon = True
dynapse_thread.start()

Exception in thread Thread-4 (dynapse_run):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/home/bmaacaron-iit.local/.virtualenvs/DynapseRA/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_40893/3980779835.py", line 898, in dynapse_run
RuntimeError: Dynap-se ID-0 SN-00000033 [3:14]: failed to set configuration parameter, modAddr=16, paramAddr=0, param=0.


In [7]:
# Example usage
"""
# Create data buffers
spike_buffer = deque(maxlen=1000)
voltage_buffer = deque(maxlen=100)

num_neurons = 10
positions = np.linspace(0, 2*pi, num_neurons, endpoint=False)

# Create plot manager
manager = DynamicPlotManager(update_interval=100)

# Add plots with method chaining
manager.add_plot(
    DynamicRasterPlot,
    data_buffer=spike_buffer,
    num_neurons=num_neurons,
    duration_window=2.0
).add_plot(
    DynamicMembraneTraces,
    data_buffer=voltage_buffer,
    neurons_to_plot=[0, 2, 4, 6, 8],
    time_window=1.0,
    Vth=-50*mV
).add_plot(
    DynamicPVAPlot,
    data_buffer=spike_buffer,
    positions=positions,
    num_neurons=num_neurons,
    time_window=2.0
)

# Setup and show plots
manager.setup().show()

# Define simulation function
def simulation():
    # Initialize simulation...
    
    # Run simulation with timesteps
    for step in range(simulation_steps):
        # Run one step
        run(dt)
        
        # Collect data
        spike_buffer.append((spikemon.i, spikemon.t))
        voltage_buffer.append((statemon.t, statemon.v))
        
        # Optional pause for visualization
        time.sleep(0.01)

# Start simulation in a separate thread
simulation_thread = threading.Thread(target=simulation)
simulation_thread.daemon = True
simulation_thread.start()
"""

'\n# Create data buffers\nspike_buffer = deque(maxlen=1000)\nvoltage_buffer = deque(maxlen=100)\n\nnum_neurons = 10\npositions = np.linspace(0, 2*pi, num_neurons, endpoint=False)\n\n# Create plot manager\nmanager = DynamicPlotManager(update_interval=100)\n\n# Add plots with method chaining\nmanager.add_plot(\n    DynamicRasterPlot,\n    data_buffer=spike_buffer,\n    num_neurons=num_neurons,\n    duration_window=2.0\n).add_plot(\n    DynamicMembraneTraces,\n    data_buffer=voltage_buffer,\n    neurons_to_plot=[0, 2, 4, 6, 8],\n    time_window=1.0,\n    Vth=-50*mV\n).add_plot(\n    DynamicPVAPlot,\n    data_buffer=spike_buffer,\n    positions=positions,\n    num_neurons=num_neurons,\n    time_window=2.0\n)\n\n# Setup and show plots\nmanager.setup().show()\n\n# Define simulation function\ndef simulation():\n    # Initialize simulation...\n\n    # Run simulation with timesteps\n    for step in range(simulation_steps):\n        # Run one step\n        run(dt)\n\n        # Collect data\n   

Error updating PVA plot: Stop value 1.893103 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 1.893103 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 1.893103 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 1.893103 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 1.893103 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 1.893103 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 1.893103 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 1.893103 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop va

Graph is destroyed while running! Note: Filter nodes constructed by `sequential` method won't work after corresponding graph is destroyed and please manually stop the graph after use.


Error updating PVA plot: Stop value 2.78102 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 2.78102 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 2.78102 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 2.78102 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 2.78102 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 2.78102 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 2.78102 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 2.78102 Hz and step value 10. ms have to have the same units. (units are Hz and s).
Error updating PVA plot: Stop value 2.78

Exception in Tkinter callback
Traceback (most recent call last):
  File "/usr/lib/python3.12/tkinter/__init__.py", line 1967, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/tkinter/__init__.py", line 864, in callit
    self.deletecommand(name)
  File "/usr/lib/python3.12/tkinter/__init__.py", line 697, in deletecommand
    self._tclCommands.remove(name)
    ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'remove'
